# Maps and Hash Tables
# =========================
**Maps** are a collection of key-value pairs, where each key is unique and maps to a specific value. They are also known as dictionaries in some programming languages. Maps provide efficient access to values based on their keys.

**Hash Tables** are a specific implementation of maps that use a hash function to compute an index into an array of buckets or slots, from which the desired value can be found. This allows for average-case constant time complexity for lookups, insertions, and deletions.


#### Question
Give a concrete implementation of the pop method in the context of the MutableMapping class, relying only on the five primary abstract methods of that class.

In [13]:
from collections.abc import MutableMapping

class MyMutableMapping(MutableMapping):
    def __init__(self, *args, **kwargs):
        self.store = dict()
        self.update(dict(*args, **kwargs))

    def __getitem__(self, key):
        return self._store[key]

    def __setitem__(self, key, value):
        self._store[key] = value

    def __delitem__(self, key):
        del self._store[key]
    
    def __len__(self):
        return len(self._store)
    
    def __iter__(self):
        return iter (self._store)
    
    def pop(self, key, default = None):
        try:
            value = self.__getitem__(key)
        except:
            if default is not None:
                return default
            raise 
        else:
            self.__delitem__(key)
            return value



#### Question
Give a concrete implementation of the items( ) method directly within the UnsortedTableMap class, ensuring that the entire iteration runs in O(n) time.

In [14]:
class UnsortedTableMap:
    """A simple Map based on an unsorted list."""

    class _Item:
        __slots__ = 'key', 'value'
        def __init__(self, key, value):
            self.key = key
            self.value = value

    def __init__(self):
        """Create an empty map."""
        self._table = []

    def __getitem__(self, k):
        for item in self._table:
            if k == item.key:
                return item.value
        raise KeyError('Key Error: ' + repr(k))

    def __setitem__(self, k, v):
        for item in self._table:
            if k == item.key:
                item.value = v
                return
        self._table.append(self._Item(k, v))

    def __delitem__(self, k):
        for j in range(len(self._table)):
            if k == self._table[j].key:
                self._table.pop(j)
                return
        raise KeyError('Key Error: ' + repr(k))

    def __len__(self):
        return len(self._table)

    def __iter__(self):
        for item in self._table:
            yield item.key

    def item(self):
        for item in self._table:
            yield (item.key, item.value)

#### Question
Draw the 11-entry hash table that results from using the hash function, $h(i) = (3i + 5)$ $mod 11$, to hash the keys 12, 44, 13, 88, 23, 94, 11, 39, 20, 16, and 5, assuming collisions are handled by chaining

#### Solution:




| Index | Value |
|-------|-------|
| 0     |    13     |
| 1     |    94 -> 39 |
| 2     |             |
| 3     |              |
| 4     |           |
| 5     |     44 -> 88 -> 11 |
| 6     |             |
| 7     |           |
| 8     |       12  -> 23  |
| 9     |    16 -> 5  |
| 10    |     20        |


#### Question
What is the result of the previous exercise, assuming collisions are handled by linear probing?
12, 44, 13, 88, 23, 94, 11, 39, 20, 16, and 5

#### Solution:

| Index | Value |
|-------|-------|
| 0     | 13            |
| 1     | 94 -> 39      |
| 2     |       23      |
| 3     |        39      |
| 4     |         5    |
| 5     | 44 -> 88 -> 11      |
| 6     |      88       |
| 7     |       11      |
| 8     | 12 -> 23            |
| 9     | 16 -> 5 |
| 10    | 20          |

Test the `ChainHashMap` class for hash tables of different types of key-value pairs and different sizes. See the Resources.zip file for an implementation of `ChainHashMap` and its superclasses.

In [15]:
'''
Created on Apr 12, 2020

@author: pglauner
'''

from collections.abc import MutableMapping
class MapBase(MutableMapping): 
    class _Item: 
        __slots__ = '_key', '_value' 
        def __init__(self, k, v): 
            self._key = k 
            self._value = v 
        def __eq__(self, other): 
            return self._key == other._key
        def __ne__(self, other): 
            return not (self == other) 
        def __lt__(self, other): 
            return self._key < other._key


class UnsortedTableMap(MapBase): 
    def __init__(self): 
        self._table = [] 
    def __getitem__(self, k): 
        for item in self._table: 
            if k == item._key: 
                return item._value 
        raise KeyError('Key Error {0}'.format(k))
    def __setitem__(self, k, v): 
        for item in self._table: 
            if k == item._key: 
                item._value = v 
                return 
        self._table.append(self._Item(k, v))  
    def __delitem__(self, k): 
        for j in range(len(self._table)): 
            if k == self._table[j]._key: 
                self._table.pop(j) 
                return 
        raise KeyError('Key Error {0}'.format(k))
    def __len__(self): 
        return len(self._table) 
    def __iter__(self): 
        for item in self._table: 
            yield item._key


In [16]:
'''
Created on Apr 12, 2020

@author: pglauner
'''
from random import randrange

class HashMapBase(MapBase):
    def __init__(self, cap=11, p=109345121):
        self._table = cap * [None]
        self._n = 0
        self._prime = p
        self._scale = 1 + randrange(p-1)
        self._shift = randrange(p)
    def _hash_function(self, k):
        return (hash(k)*self._scale + self._shift) \
                    % self._prime % len(self._table)
    def __len__(self):
        return self._n
    def __getitem__(self, k):
        j = self._hash_function(k)
        return self._bucket_getitem(j, k)
    def __setitem__(self, k, v):
        j = self._hash_function(k)
        self._bucket_setitem(j, k, v)
        if self._n > len(self._table) // 2:
            self._resize(2 * len(self._table) - 1)
    def __delitem__(self, k):
        j = self._hash_function(k)
        self._bucket_delitem(j, k)
        self._n -= 1
    def _resize(self, c):
        old = list(self.items())
        self._table = c * [None]
        self._n = 0
        for (k, v) in old:
            self[k] = v


class ChainHashMap(HashMapBase):
    def _bucket_getitem(self, j, k):
        bucket = self._table[j]
        if bucket is None:
            raise KeyError('Key Error {0}'.format(k))
        return bucket[k]
    def _bucket_setitem(self, j, k, v):
        if self._table[j] is None:
            self._table[j] = UnsortedTableMap()
        oldsize = len(self._table[j])
        self._table[j][k] = v
        if len(self._table[j])>oldsize:
            self._n += 1
    def _bucket_delitem(self, j, k):
        bucket = self._table[j]
        if bucket is None:
            raise KeyError('Key Error {0}'.format(k))
        del bucket[k]
    def __iter__(self):
        for bucket in self._table:
            if bucket is not None:
                yield from bucket

In [17]:
def print_map(m):
    print("Map size:", len(m))
    print("Contents:")
    for k in m:
        print(f"  {k!r}: {m[k]!r}")
    print()

def test_int_str():
    print("=== test_int_str ===")
    m = ChainHashMap()
    m[1] = "one"
    m[2] = "two"
    m[3] = "three"
    print_map(m)
    assert m[2] == "two"
    m[2] = "TWO"
    assert m[2] == "TWO"
    del m[1]
    print_map(m)

def test_str_tuple():
    print("=== test_str_tuple ===")
    m = ChainHashMap()
    m["key"] = (1, 2)
    m["other"] = (3, 4)
    print_map(m)
    assert m["key"] == (1, 2)

def test_tuple_float():
    print("=== test_tuple_float ===")
    m = ChainHashMap()
    m[(0, 0)] = 0.0
    m[(1, 2)] = 3.14
    print_map(m)
    assert m[(1, 2)] == 3.14

def test_resize():
    print("=== test_resize ===")
    m = ChainHashMap(cap=3)
    for i in range(20):
        m[i] = i*i
    print_map(m)
    assert m[10] == 100
    assert len(m) == 20

def test_missing_key():
    print("=== test_missing_key ===")
    m = ChainHashMap()
    try:
        _ = m["notfound"]
    except KeyError:
        print("Correctly raised KeyError")
    print()

def test_collisions():
    print("=== test_collisions ===")
    m = ChainHashMap(cap=2)
    m["a"] = 1
    m["b"] = 2
    m["c"] = 3
    print_map(m)

if __name__ == "__main__":
    test_int_str()
    test_str_tuple()
    test_tuple_float()
    test_resize()
    test_missing_key()
    test_collisions()

=== test_int_str ===
Map size: 3
Contents:
  2: 'two'
  3: 'three'
  1: 'one'

Map size: 2
Contents:
  2: 'TWO'
  3: 'three'

=== test_str_tuple ===
Map size: 2
Contents:
  'other': (3, 4)
  'key': (1, 2)

=== test_tuple_float ===
Map size: 2
Contents:
  (1, 2): 3.14
  (0, 0): 0.0

=== test_resize ===
Map size: 20
Contents:
  15: 225
  16: 256
  14: 196
  17: 289
  5: 25
  6: 36
  7: 49
  8: 64
  18: 324
  19: 361
  9: 81
  10: 100
  11: 121
  12: 144
  13: 169
  0: 0
  1: 1
  2: 4
  3: 9
  4: 16

=== test_missing_key ===
Correctly raised KeyError

=== test_collisions ===
Map size: 3
Contents:
  'b': 2
  'c': 3
  'a': 1



#### Question
```python
class HashMapBase ( MapBase ):
    def __init__(self , cap =11, p =109345121) :
    self. table = cap ∗ [None]
    self. n = 0
    self. prime = p
    self. scale = 1 + randrange (p−1)
    self. shift = randrange (p)
    def hash_function(self, k):
        return (hash(k) * self.scale + self.shift) % self.prime % len(self.table)

    def __len__(self):
        return self.n

    def __getitem__(self, k):
        j = self.hash_function(k)
        return self.bucket_getitem(j, k)

    def __setitem__(self, k, v):
        j = self.hash_function(k)
        self.bucket_setitem(j, k, v)
        if self.n > len(self.table) // 2:
            self.resize(2 * len(self.table) - 1)

    def __delitem__(self, k):
        j = self.hash_function(k)
        self.bucket_delitem(j, k)
        self.n -= 1

    def resize(self, c):
        old = list(self.items())
        self.table = c * [None]
        self.n = 0
        for (k, v) in old:
            self[k] = v

```
Review the generic hash table implementation in Algorithm. Explain why attribute n is updated in 
```python 
delitem (self, k)
``` 
but not updated in 
```python
setitem (self, k, v)
``` 

#### Solution:
In the `delitem(self, k)` method, the attribute `n` is decremented by 1 to reflect the removal of an item from the hash table. This is necessary because the `n` attribute keeps track of the number of items currently stored in the hash table, and removing an item reduces that count. In contrast, the `setitem(self, k, v)` method does not change the number of items in the table when an item is added or updated, so the `n` attribute is not modified in that case.